# Lenguaje de Programación Visual - FIUNA
## Semana 3: Concurrencia y Persistencia Temporal (Asyncio + SQLite)

## 1. Fundamentos de Asincronía con Asyncio

En mecatrónica, muchas tareas son **I/O Bound** (espera de datos por puerto serie, sockets de red, lectura/escritura en disco). En lugar de congelar la ejecución usando hilos pesados o bloqueos síncronos (`time.sleep`), `asyncio` permite cooperatividad no bloqueante sobre un **Event Loop** (bucle de eventos).

- `async def`: Define una corrutina.
- `await`: Pausa la corrutina actual y cede el control del bucle de eventos a otra tarea mientras se espera un evento I/O.
- `asyncio.gather()`: Ejecuta múltiples corrutinas concurrentemente.

In [ ]:
import asyncio
import time

async def simular_lectura_sensor(nombre: str, latencia: float):
    print(f"[{time.strftime('%H:%M:%S')}] Iniciando lectura de {nombre}...")
    await asyncio.sleep(latencia)  # Cede el Event Loop
    print(f"[{time.strftime('%H:%M:%S')}] Lectura completada en {nombre} ({latencia}s)")
    return f"{nombre}_OK"

async def main_ejemplo1():
    t0 = time.time()
    # Ejecutar 3 lecturas en paralelo
    resultados = await asyncio.gather(
        simular_lectura_sensor("Sensor_A", 1.0),
        simular_lectura_sensor("Sensor_B", 0.5),
        simular_lectura_sensor("Sensor_C", 1.5)
    )
    tf = time.time()
    print(f"Resultados: {resultados}")
    print(f"Tiempo total transcurrido: {tf - t0:.2f} segundos (vs 3.0s síncronos)")

# Ejecutar dentro de un entorno asíncrono o con await en Jupyter
await main_ejemplo1()

### Patrón Productor-Consumidor con `asyncio.Queue`

Cuando múltiples sensores emiten datos a frecuencias heterogéneas, la inserción inmediata en base de datos genera cuellos de botella por I/O de disco. `asyncio.Queue` actúa como un buffer en memoria intermedia.

In [ ]:
async def productor(queue: asyncio.Queue, id_productor: int, cantidad: int):
    for i in range(cantidad):
        item = f"Dato_{id_productor}_{i+1}"
        await queue.put(item)
        await asyncio.sleep(0.1)
    print(f"Productor {id_productor} finalizado.")

async def consumidor(queue: asyncio.Queue, id_consumidor: int, total_esperado: int):
    procesados = 0
    while procesados < total_esperado:
        item = await queue.get()
        procesados += 1
        print(f"[Consumidor {id_consumidor}] Procesado: {item}")
        queue.task_done()

async def demo_queue():
    cola = asyncio.Queue()
    # 2 productores de 3 elementos = 6 elementos totales
    await asyncio.gather(
        productor(cola, 1, 3),
        productor(cola, 2, 3),
        consumidor(cola, 100, 6)
    )

await demo_queue()

## 2. Persistencia Temporal con SQLite y WAL Mode

SQLite es la base de datos embebida estándar en la industria. Sin embargo, en su configuración predeterminada (journal mode = DELETE), cada `COMMIT` bloquea toda la base de datos para escrituras y lecturas.

### Modo WAL (Write-Ahead Logging):
- **Ventaja**: Las lecturas y escrituras ocurren concurrentemente sin bloquearse.
- Las escrituras se almacenan temporalmente en un archivo `.db-wal` antes de consolidarse en el archivo principal `.db`.
- **PRAGMA synchronous = NORMAL**: Reduce las operaciones síncronas de fsync a disco preservando integridad ACID.

In [ ]:
import aiosqlite
import os

DB_TEST = "notebook_telemetry.db"

if os.path.exists(DB_TEST):
    os.remove(DB_TEST)

async def demo_sqlite_wal():
    async with aiosqlite.connect(DB_TEST) as db:
        # Habilitar WAL Mode
        async with db.execute("PRAGMA journal_mode = WAL;") as cursor:
            modo = await cursor.fetchone()
            print(f"Modo de Journal configurado a: {modo[0]}")
        
        await db.execute("PRAGMA synchronous = NORMAL;")

        # Crear tabla de series temporales
        await db.execute("""
        CREATE TABLE IF NOT EXISTS mediciones (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp_unix REAL NOT NULL,
            sensor TEXT NOT NULL,
            valor REAL NOT NULL
        );
        """)

        await db.execute("CREATE INDEX IF NOT EXISTS idx_sensor_time ON mediciones(sensor, timestamp_unix);")
        await db.commit()

        # Inserción por lotes (Batch Insert)
        lote = [
            (time.time() + i, "TEMP_ENGINE", 70.0 + i * 0.5)
            for i in range(10)
        ]

        await db.executemany("INSERT INTO mediciones (timestamp_unix, sensor, valor) VALUES (?, ?, ?);", lote)
        await db.commit()
        print(f"Insertados {len(lote)} registros en batch exitosamente.")

await demo_sqlite_wal()

### Consultas de Agregación Temporal

In [ ]:
async def consultar_metricas():
    async with aiosqlite.connect(DB_TEST) as db:
        db.row_factory = aiosqlite.Row
        async with db.execute("""
            SELECT sensor, COUNT(*) as total, AVG(valor) as prom, MIN(valor) as min_val, MAX(valor) as max_val
            FROM mediciones
            GROUP BY sensor;
        """) as cursor:
            filas = await cursor.fetchall()
            for row in filas:
                print(f"Sensor: {row['sensor']} | Total: {row['total']} | Promedio: {row['prom']:.2f} | Mín: {row['min_val']} | Máx: {row['max_val']}")

await consultar_metricas()

## 3. Pipeline Mecatrónico Completo y Visualización

A continuación, consultamos los datos almacenados en SQLite y graficamos los resultados usando `matplotlib`.

In [ ]:
import matplotlib.pyplot as plt

async def graficar_telemetria():
    async with aiosqlite.connect(DB_TEST) as db:
        db.row_factory = aiosqlite.Row
        async with db.execute("SELECT timestamp_unix, sensor, valor FROM mediciones ORDER BY timestamp_unix ASC;") as cursor:
            rows = await cursor.fetchall()
            data = [dict(r) for r in rows]

    if data:
        timestamps = [d['timestamp_unix'] for d in data]
        valores = [d['valor'] for d in data]
        plt.figure(figsize=(10, 4))
        plt.plot(timestamps, valores, marker='o', color='b', label='TEMP_ENGINE')
        plt.title('Persistencia Temporal de Telemetría (SQLite WAL)')
        plt.xlabel('Timestamp Unix (s)')
        plt.ylabel('Temperatura (°C)')
        plt.grid(True)
        plt.legend()
        plt.show()
    else:
        print('No se encontraron datos para graficar.')

await graficar_telemetria()

---
### Limpieza de Archivos Temporales
Al finalizar las pruebas del notebook, eliminamos el archivo `.db` generado en esta sesión.

In [ ]:
if os.path.exists(DB_TEST):
    os.remove(DB_TEST)
    print(f"Base de datos temporal '{DB_TEST}' eliminada de forma limpia.")